Part 2:Indexing and Evaluation

1-Importing libraries and Data

We first start by importing the libraries and the data 

In [1]:
# if you do not have 'nltk', the following command should work "python -m pip install nltk"
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ADMIN\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
from collections import defaultdict
from array import array #stores a sequence of values of a specified type
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import math
import numpy as np
import collections
import pandas as pd
from pathlib import Path
from ast import literal_eval # converts list representation bacfk into python lists
from numpy import linalg as la

fashion_products.csv retrieves the original product information using its ID.

index_info.csv build the inverted index using the preprocessed words.

In [7]:

import pandas as pd
from pathlib import Path

data_dir = Path("data")

# Load original product information
df = pd.read_csv(data_dir / "fashion_products.csv")

# Load the preprocessed data from Part 1
index_df = pd.read_csv(data_dir / "index_info.csv")

print("Original dataset:", df.shape)
print("Preprocessed dataset:", index_df.shape)

display(index_df.head())


Original dataset: (28080, 17)
Preprocessed dataset: (28080, 8)


,pid,title,description,brand,category,sub_category,seller,product_details
0,TKPFCZ9EA7H5FYZH,"['solid', 'women', 'multicolor', 'track', 'pant']","['yorker', 'trackpant', 'made', '100', 'rich',...",['york'],"['cloth', 'accessori']",['bottomwear'],"['shyam', 'enterpris']","['1005combo2', 'elast', 'side', 'pocket', 'cot..."
1,TKPFCZ9EJZV2UVRZ,"['solid', 'men', 'blue', 'track', 'pant']","['yorker', 'trackpant', 'made', '100', 'rich',...",['york'],"['cloth', 'accessori']",['bottomwear'],"['shyam', 'enterpris']","['1005blue', 'drawstr', 'elast', 'side', 'pock..."
2,TKPFCZ9EHFCY5Z4Y,"['solid', 'men', 'multicolor', 'track', 'pant']","['yorker', 'trackpant', 'made', '100', 'rich',...",['york'],"['cloth', 'accessori']",['bottomwear'],"['shyam', 'enterpris']","['1005combo4', 'elast', 'side', 'pocket', 'cot..."
3,TKPFCZ9ESZZ7YWEF,"['solid', 'women', 'multicolor', 'track', 'pant']","['yorker', 'trackpant', 'made', '100', 'rich',...",['york'],"['cloth', 'accessori']",['bottomwear'],"['shyam', 'enterpris']","['1005combo3', 'elast', 'side', 'pocket', 'cot..."
4,TKPFCZ9EVXKBSUD7,"['solid', 'women', 'brown', 'grey', 'track', '...","['yorker', 'trackpant', 'made', '100', 'rich',...",['york'],"['cloth', 'accessori']",['bottomwear'],"['shyam', 'enterpris']","['1005combo1', 'drawstr', 'elast', 'side', 'po..."


Part 1: Indexing:

Building the inverted index:

The inverted index that will be implemented will follow the vocabulary data structure seen in class:

{
Term_id_1: [document_1, document_2, document_4],
Term_id_2: [document_1, document_3, document_5, document_6],
etc…
}

Example: 
{
    "jeans": ["P001", "P002", "P004"],
    "multicolor": ["P001", "P003"]
}

However we decided to extend this simple structure by creating a field-aware positional inverted index. This means that additionaly to storing product pids, our implementation will record the field in which each term appears the its position within this field.

Therefore our structure would follow this form:

{
    "blue": {
        "title": [("P001", [0]), ("P002", [2])],
        "description": [("P004", [5])]
    }
}

To construct this index we iterate through the 7 fields of the preprocessed text fields. For each field we collect the positions of each term and then add the corresponding postinf to the inverted index.

This approach preserved the functionality of the basic inverted index introduces in class in addition to providing information supporting term-freqency calculations and field-specific retrieval.

In [13]:

from collections import defaultdict
from ast import literal_eval

def build_inverted_index(index_df):
    """
    Build a positional inverted index from the preprocessed products.

    Parameters:
        index_df: DataFrame containing product IDs and tokenized fields.

    Returns:
        inverted_index: dictionary mapping each term to its fields,
                        product IDs, and positions.
    """

    inverted_index = defaultdict(lambda: defaultdict(list))

    searchable_fields = [
        "title", "description", "brand",
        "category", "sub_category",
        "seller", "product_details"
    ]

    # Go through each product in the dataset
    for product in index_df.to_dict("records"):

        product_id = product["pid"]

        # Process each text field separately
        for field in searchable_fields:

            tokens = product[field]

            # Convert CSV strings back to token lists if necessary
            if isinstance(tokens, str):
                tokens = literal_eval(tokens)

            if not isinstance(tokens, list):
                continue

            # Record the positions of each term in this field
            term_positions = defaultdict(list)

            for position, term in enumerate(tokens):
                term_positions[term].append(position)

            # Add the field's postings to the inverted index
            for term, positions in term_positions.items():
                inverted_index[term][field].append(
                    (product_id, positions)
                )

    return inverted_index


# Build the index
inverted_index = build_inverted_index(index_df)



In [37]:
#We chose these queries:
# Our five proposed test queries
test_queries = [
    "Women grinch printed pyjamas",
    "Blue navy jeans for women",
    "Men dotted tie",
    "Women silk dress",
    "Men brown winter pants"
]

# Display the selected queries
for i, query in enumerate(test_queries, start=1):
    print(f"Query {i}: {query}")


Query 1: Women grinch printed pyjamas
Query 2: Blue navy jeans for women
Query 3: Men dotted tie
Query 4: Women silk dress
Query 5: Men brown winter pants


### Query Preprocessing

Before searching the inverted index, we preprocess our five selected queries using the same steps applied to the products in Part 1: lowercasing, tokenization, stop-word removal, and stemming.

In [38]:


from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

tokenizer = RegexpTokenizer(r"[a-z0-9]+")

def build_terms(query):
    """
    Preprocess a search query by removing stop words,
    applying stemming, and converting text to lowercase.

    Argument:
    query -- string containing the search query

    Returns:
    query -- list of preprocessed query terms
    """

    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))

    query = query.lower()  # Transform to lowercase

    query = tokenizer.tokenize(query)  # Tokenize the query

    query = [word for word in query if word not in stop_words]  # Remove stopwords

    query = [stemmer.stem(word) for word in query]  # Apply stemming

    return query


Testing the Search Engine

After implementing preprocessing the querie, we test our five selected queries against the inverted index.
For each query, we will examine three things:

1-Its preprocessed terms
2-Absent terms from the vocabulary 
3-We will count the products satisfying the AND condition.

 We will also display an example product to inspect the retrieved results.

This helps us assess whether our queries are suitable for subsequent TF-IDF ranking and evaluation.

In [39]:

# Create a dictionary to retrieve original product titles
product_titles = dict(zip(df["pid"], df["title"]))

# Test the five queries
for number, query in enumerate(test_queries, start=1):

    terms = build_terms(query)
    missing_terms = [term for term in terms
                     if term not in inverted_index]

    results = search(query, inverted_index)

    print(f"\nQuery {number}: {query}")
    print("Processed terms:", terms)
    print("Missing terms:", missing_terms)
    print("Matching products:", len(results))

    # Display one example if results exist
    if results:
        first_pid = results[0]
        print("Example product:", product_titles[first_pid])
    else:
        print("No matching products found.")

    print("-" * 50)



Query 1: Women grinch printed pyjamas
Processed terms: ['women', 'grinch', 'print', 'pyjama']
Missing terms: ['grinch']
Matching products: 0
No matching products found.
--------------------------------------------------

Query 2: Blue navy jeans for women
Processed terms: ['blue', 'navi', 'jean', 'women']
Missing terms: []
Matching products: 43
Example product: Slim Women Dark Blue Jeans
--------------------------------------------------

Query 3: Men dotted tie
Processed terms: ['men', 'dot', 'tie']
Missing terms: []
Matching products: 3
Example product: nu-Lite Satin Tie & Cufflink  (Blue)
--------------------------------------------------

Query 4: Women silk dress
Processed terms: ['women', 'silk', 'dress']
Missing terms: []
Matching products: 32
Example product: nu-Lite Satin Tie Pin Set  (Yellow)
--------------------------------------------------

Query 5: Men brown winter pants
Processed terms: ['men', 'brown', 'winter', 'pant']
Missing terms: []
Matching products: 0
No matchin

As we saw from our previous result, 4 out of the 5 queries we suggested have no matching products, for this reason we will test simpler versions of the queries 

In [41]:

# Test simpler versions of queries that returned no results

alternative_queries = [
    "women pyjamas",
    "women silk dress",
    "men brown pant"
]

for query in alternative_queries:
    results = search(query, inverted_index)

    print("Query:", query)
    print("Processed terms:", build_terms(query))
    print("Matching products:", len(results))
    print("-" * 40)


Query: women pyjamas
Processed terms: ['women', 'pyjama']
Matching products: 193
----------------------------------------
Query: women silk dress
Processed terms: ['women', 'silk', 'dress']
Matching products: 32
----------------------------------------
Query: men brown pant
Processed terms: ['men', 'brown', 'pant']
Matching products: 56
----------------------------------------



### Query Selection and Refinement

We initially proposed five creative and specific search queries to test the retrieval system. However, our first testing revealed that three of these queries returned no matching products.

This was mainly due to the strict AND retrieval requirement. For example, the term "grinch" was absent from our vocabulary, while other queries contained combinations of terms that did not occur together in any product.

We therefore tested simpler versions of these queries while preserving their original shopping intentions. This allowed us to obtain more meaningful retrieval results for the subsequent TF-IDF ranking and evaluation.

This little experiment however also highlights a limitation of conjunctive retrieval: adding more specific terms can significantly reduce the number of matching documents.

In [42]:

# Final five queries selected for ranking and evaluation
final_queries = [
    "Women pyjamas",
    "Blue navy jeans for women",
    "Men dotted tie",
    "Women silk dress",
    "Men brown pant"
]

# Verify the final queries
for i, query in enumerate(final_queries, start=1):
    results = search(query, inverted_index)

    print(f"Query {i}: {query}")
    print(f"Matching products: {len(results)}")
    print("-" * 40)


Query 1: Women pyjamas
Matching products: 193
----------------------------------------
Query 2: Blue navy jeans for women
Matching products: 43
----------------------------------------
Query 3: Men dotted tie
Matching products: 3
----------------------------------------
Query 4: Women silk dress
Matching products: 32
----------------------------------------
Query 5: Men brown pant
Matching products: 56
----------------------------------------


So far, our search() function has retrieved matching products ID, eg:193 products for Women pyjamas. However this function does not mention which of these 193 documents is most relevant. 
This leads us to the final part of Part 1 which would be ranking our results.


### Term Frequency (TF)

TF measures how many times a term appears in a document.
We will calculate the frequency of each term across the sven indexed fields of a product.

In [ ]:

from collections import defaultdict, Counter

def calculate_tf(inverted_index):
    """
    Calculate the frequency of each term in every product.

    Argument:
    inverted_index -- field-aware positional inverted index

    Returns:
    tf -- dictionary containing term frequencies per product
    """

    tf = defaultdict(Counter)

    # Iterate through every indexed term
    for term, field_dict in inverted_index.items():

        # Look at every field containing that term
        for postings in field_dict.values():

            # Count occurrences for each product
            for pid, positions in postings:
                tf[pid][term] += len(positions)
                #rf[pid] selects our product
                #[term] select the word
                #len(positions) counts the number of times a word appears in the field
                 
    return tf


# Calculate term frequencies
tf = calculate_tf(inverted_index)

print("Number of products with TF values:", len(tf))


Number of products with TF values: 28080


In [44]:

# Select our first query: Women pyjamas
query = final_queries[0]

# Retrieve the 193 matching products
results = search(query, inverted_index)

print("Query:", query)
print("Number of matching products:", len(results))

# Select the first matching product
pid = results[0]

print("\nProduct ID:", pid)
print("Product title:",
      df.loc[df["pid"] == pid, "title"].iloc[0])

# Display term frequencies for this product
print("\nTerm frequencies:")

for term in build_terms(query):
    print(f"{term}: {tf[pid][term]}")


Query: Women pyjamas
Number of matching products: 193

Product ID: ETHF25572QHXAGEQ
Product title: Women Kurta and Pyjama Set Cotton Blend

Term frequencies:
women: 1
pyjama: 2


In [45]:

pid = "ETHF25572QHXAGEQ"

for term in ["women", "pyjama"]:
    print("\nTerm:", term)

    for field, postings in inverted_index[term].items():
        for product_id, positions in postings:
            if product_id == pid:
                print(f"Field: {field}, Positions: {positions}")



Term: women
Field: title, Positions: [0]

Term: pyjama
Field: title, Positions: [2]
Field: product_details, Positions: [3]


### Document Frequency (DF)

Document Frequency measures the number of distinct products containing a term, regardless of how many times or in which fields it appears.

In [46]:

def calculate_df(inverted_index):
    """
    Calculate the document frequency of each indexed term.

    Returns:
        df_counts -- number of distinct products containing each term
    """

    df_counts = {}

    for term, fields in inverted_index.items():

        product_ids = set()

        for postings in fields.values():
            for pid, positions in postings:
                product_ids.add(pid)

        df_counts[term] = len(product_ids)

    return df_counts


# Calculate document frequencies
df_counts = calculate_df(inverted_index)

# Examine the terms from our first query
print("DF of women:", df_counts["women"])
print("DF of pyjama:", df_counts["pyjama"])


DF of women: 13817
DF of pyjama: 362


### Inverse Document Frequency (IDF)

IDF measures how informative a term is across the product
collection. Terms appearing in fewer products receive
higher IDF values, while common terms receive lower values.

In [48]:

import math

def calculate_idf(df_counts, total_products):
    """
    Calculate IDF for every term in the vocabulary.
    """

    idf = {}

    for term, frequency in df_counts.items():
        idf[term] = math.log(total_products / frequency)

    return idf


# Total number of products
total_products = index_df["pid"].nunique()

# Calculate IDF
idf = calculate_idf(df_counts, total_products)

print("IDF of women:", round(idf["women"], 4))
print("IDF of pyjama:", round(idf["pyjama"], 4))


IDF of women: 0.7092
IDF of pyjama: 4.3512


As expected, we got a higher IDF for pyjama than women, because "pyjama" appears in far fewer products

### TF-IDF Calculation

We calculate TF-IDF by multiplying the term frequency (TF)
of each term in a product by its inverse document frequency (IDF).

For each query, the TF-IDF weights of its terms are summed to
obtain a relevance score for the product.

In [49]:

def rank_tfidf(query, inverted_index, tf, idf):
    """
    Rank all products matching a query using TF-IDF.

    Returns:
        List of (product ID, TF-IDF score) pairs,
        ordered from highest to lowest score.
    """

    # Retrieve all products satisfying AND retrieval
    matching_products = search(query, inverted_index)

    # Preprocess the query
    query_terms = set(build_terms(query))

    ranked_products = []

    # Calculate a score for every matching product
    for pid in matching_products:

        score = 0

        for term in query_terms:
            score += tf[pid][term] * idf[term]

        ranked_products.append((pid, score))

    # Sort by decreasing TF-IDF score
    ranked_products.sort(key=lambda item: (-item[1], item[0]))

    return ranked_products


### Normalized Term Frequency

Following the approach introduced in the practical lab, we normalize
raw term frequencies using the Euclidean norm of each product's term
frequency vector.

The norm is calculated using all indexed terms of the product.
This reduces the effect of differences in document length.

In [50]:

import math

def normalize_tf(tf):
    """
    Normalize the term frequencies of each product
    using the Euclidean norm.

    Argument:
    tf -- raw term frequencies for each product

    Returns:
    normalized_tf -- normalized term frequencies
                     for every product and term
    """

    normalized_tf = {}

    for pid, term_frequencies in tf.items():

        # Calculate the normalization factor
        norm = 0

        for term, frequency in term_frequencies.items():
            norm += frequency ** 2

        norm = math.sqrt(norm)

        # Normalize each term frequency
        normalized_tf[pid] = {}

        if norm > 0:
            for term, frequency in term_frequencies.items():
                normalized_tf[pid][term] = frequency / norm

    return normalized_tf


# Normalize TF for every indexed product
normalized_tf = normalize_tf(tf)

print("Products with normalized TF:", len(normalized_tf))


Products with normalized TF: 28080


In [51]:

pid = "ETHF25572QHXAGEQ"

for term in ["women", "pyjama"]:
    print("Term:", term)
    print("Raw TF:", tf[pid][term])
    print("Normalized TF:", round(normalized_tf[pid][term], 4))
    print()


Term: women
Raw TF: 1
Normalized TF: 0.0685

Term: pyjama
Raw TF: 2
Normalized TF: 0.137



In [52]:

from collections import Counter
import math

def rank_documents(query, index, normalized_tf, idf):
    """
    Rank products using TF-IDF vector weights.

    The function only ranks products satisfying
    conjunctive (AND) retrieval.
    """

    # Preprocess the query
    query_terms = build_terms(query)

    # Retrieve products matching ALL query terms
    matching_products = search(query, index)

    if not query_terms or not matching_products:
        return []

    # Calculate the frequency of terms in the query
    query_counts = Counter(query_terms)

    # Normalize the query term frequencies
    query_norm = math.sqrt(
        sum(count ** 2 for count in query_counts.values())
    )

    # Create the TF-IDF query vector
    query_weights = {}

    for term, count in query_counts.items():
        query_tf = count / query_norm
        query_weights[term] = query_tf * idf[term]

    # Calculate a relevance score for each matching product
    ranked_products = []

    for pid in matching_products:

        score = 0

        for term, query_weight in query_weights.items():

            # TF-IDF weight of this term in the product
            document_weight = normalized_tf[pid][term] * idf[term]

            # Dot product contribution
            score += document_weight * query_weight

        ranked_products.append((pid, score))

    # Sort products by decreasing relevance
    ranked_products.sort(key=lambda item: (-item[1], item[0]))

    return ranked_products


In [53]:

query = final_queries[0]

ranked_results = rank_documents(
    query, inverted_index, normalized_tf, idf
)

print("Query:", query)
print("Number of ranked products:", len(ranked_results))

print("\nTop 10 ranked products:")

product_titles = dict(zip(df["pid"], df["title"]))

for pid, score in ranked_results[:10]:
    print(f"{product_titles[pid]} | Score: {score:.4f}")


Query: Women pyjamas
Number of ranked products: 193

Top 10 ranked products:
Women Pyjama  (Pack of 1) | Score: 8.2949
Women Pyjama  (Pack of 1) | Score: 8.2949
Women Pyjama  (Pack of 1) | Score: 8.2949
Women Pyjama  (Pack of 1) | Score: 8.2949
Women Pyjama  (Pack of 1) | Score: 8.2757
Women Pyjama  (Pack of 1) | Score: 8.2757
Women Pyjama  (Pack of 1) | Score: 8.2757
Women Pyjama  (Pack of 1) | Score: 7.4285
Women Pyjama  (Pack of 1) | Score: 7.4285
Women Pyjama  (Pack of 1) | Score: 7.4285


In [54]:

print("Top 10 ranked products:\n")

for rank, (pid, score) in enumerate(ranked_results[:10], start=1):
    product = df[df["pid"] == pid].iloc[0]

    print(f"Rank {rank}")
    print("PID:", pid)
    print("Title:", product["title"])
    print("Brand:", product["brand"])
    print("TF-IDF Score:", round(score, 4))
    print("-" * 40)

print("\nUnique product IDs:", len(set(pid for pid, score in ranked_results)))
print("Total ranked products:", len(ranked_results))


Top 10 ranked products:

Rank 1
PID: PYJFZMJN25GEGHQE
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 8.2949
----------------------------------------
Rank 2
PID: PYJFZMJNGEDZGHEZ
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 8.2949
----------------------------------------
Rank 3
PID: PYJFZMJNHZF3ZYNR
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 8.2949
----------------------------------------
Rank 4
PID: PYJFZMJNRQDEFBHB
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 8.2949
----------------------------------------
Rank 5
PID: PYJFZMJNAWFWGEGA
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 8.2757
----------------------------------------
Rank 6
PID: PYJFZMJNFPHYWZN7
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 8.2757
----------------------------------------
Rank 7
PID: PYJFZMJNGNZ6GMKZ
Title: Women Pyjama  (Pack of 1)
Brand: Bonnevil
TF-IDF Score: 8.2757
------------------------------------

### Indexing and Ranking Methodology

We implemented a field-aware positional inverted index using the
preprocessed product data from Part 1. Each term is associated with
the product IDs, fields, and positions where it appears.

For retrieval, we implemented conjunctive (AND) search, ensuring
that all query terms occur in each retrieved product.

For ranking, we adapted the TF-IDF vector-space approach introduced
in the practical lab. We calculated normalized term frequencies,
document frequencies, and inverse document frequencies across the
product collection.

Both query and document terms were assigned TF-IDF weights, and
relevance scores were computed using their dot product. Retrieved
products were then sorted in descending order of relevance.

We tested the search engine using five queries with different
product characteristics and observed both relevant results and
some limitations caused by matching terms across multiple fields.